# Customer Churn Prediction – Reproducible Pipeline
**Project:** A Reproducible and Fairness-Aware Machine Learning Pipeline for Customer Churn Prediction in a Subscription Service  
**Author:** Valerio Gomez  
**Date:** June 2026  
**Course:** UNMSM – Research Methods & Scientific Integrity in AI

## 1. Environment Setup & Reproducibility

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, recall_score, f1_score, confusion_matrix

import mlflow
import mlflow.sklearn

import warnings
warnings.filterwarnings('ignore')

# --- Reproducibility Setup ---
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

print('Reproducibility baseline established. SEED =', SEED)

## 2. Load Dataset

In [ ]:
# Load the dataset
# Path assumes execution from 05_pipeline directory
df = pd.read_csv('../data/churn_dataset.csv')
print(f"Dataset Shape: {df.shape}")
df.head()

## 3. Exploratory Data Analysis (EDA)
Let us explore demographic distributions, check for class imbalances, examine features correlation, and analyze potential gender-based representation differences.

### 3.1 Class Balance & Summary Statistics

In [ ]:
# General summary statistics
print(df.describe())

# Missing values check
print("\nMissing values per column:\n", df.isnull().sum())

# Churn target balance check
plt.figure(figsize=(6, 4))
sns.countplot(x='Churn', data=df, palette='Set2')
plt.title('Distribution of Target Variable (Churn)')
plt.xlabel('Churn (0 = Retained, 1 = Churned)')
plt.ylabel('Count')
plt.show()

churn_rate = df['Churn'].mean() * 100
print(f"Overall Churn Rate: {churn_rate:.2f}%")

### 3.2 Distributions of Numerical Features

In [ ]:
num_cols = ['Age', 'Monthly_Spending', 'Subscription_Length', 'Support_Interactions']

# Plot histograms with KDE for numerical features
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for i, col in enumerate(num_cols):
    ax = axes[i//2, i%2]
    sns.histplot(data=df, x=col, hue='Churn', kde=True, ax=ax, palette='coolwarm', multiple='stack')
    ax.set_title(f'Distribution of {col} by Churn Status')
    ax.set_xlabel(col)
    ax.set_ylabel('Count')

plt.tight_layout()
plt.show()

### 3.3 Outlier Detection & Target Relationships

In [ ]:
# Boxplots to inspect feature spread, medians, and outliers across Churn classes
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for i, col in enumerate(num_cols):
    ax = axes[i//2, i%2]
    sns.boxplot(data=df, x='Churn', y=col, ax=ax, palette='Set3')
    ax.set_title(f'{col} Boxplot by Churn Status')
    ax.set_xlabel('Churn')
    ax.set_ylabel(col)

plt.tight_layout()
plt.show()

### 3.4 Bivariate Analysis & Correlation Matrix

In [ ]:
# Correlation Matrix Heatmap (excluding CustomerID)
plt.figure(figsize=(8, 6))
corr_matrix = df.drop('Customer_ID', axis=1).corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('Correlation Matrix of Churn Predictors')
plt.show()

### 3.5 Gender Representation & Bias Baseline Analysis
Here, we perform a preliminary fairness audit on the raw historical data to verify if the churn rate varies significantly across the gender attribute (protected feature).

In [ ]:
# Count of samples by Gender
print("Sample counts by Gender (0 = Female, 1 = Male):")
print(df['Gender'].value_counts())

# Calculate Churn Rate per Gender group
gender_churn = df.groupby('Gender')['Churn'].mean().reset_index()
gender_churn.columns = ['Gender', 'Churn_Rate']
print("\nChurn Rate by Gender:")
print(gender_churn)

# Visualizing churn distribution across Gender
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Countplot
sns.countplot(x='Gender', hue='Churn', data=df, ax=axes[0], palette='Set1')
axes[0].set_title('Churn Counts by Gender')
axes[0].set_xlabel('Gender (0 = Female, 1 = Male)')
axes[0].set_ylabel('Count')

# Barplot of rates
sns.barplot(x='Gender', y='Churn_Rate', data=gender_churn, ax=axes[1], palette='pastel')
axes[1].set_title('Churn Proportion (Rate) by Gender')
axes[1].set_xlabel('Gender (0 = Female, 1 = Male)')
axes[1].set_ylabel('Churn Rate')
axes[1].set_ylim(0, 1.0)

plt.show()

## 4. Preprocessing: Train-Test Split & Scaling

In [ ]:
# Preprocessing: split FIRST, then scale
X = df.drop(['Customer_ID', 'Churn'], axis=1)
y = df['Churn']

# Stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)

# Scale on training data ONLY to prevent leakage
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Train size: {X_train.shape}, Test size: {X_test.shape}')

## 5. Model Selection & Experiments (MLflow Tracking)

In [ ]:
# Define models and train with MLflow tracking
models = {
    'Logistic Regression': LogisticRegression(random_state=SEED, max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=SEED),
    'Random Forest': RandomForestClassifier(random_state=SEED),
    'Gradient Boosting': GradientBoostingClassifier(random_state=SEED),
    'SVM': SVC(random_state=SEED)
}

mlflow.set_experiment('Churn_Experiment')

results = []
for name, model in models.items():
    with mlflow.start_run(run_name=name):
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)

        acc = accuracy_score(y_test, y_pred)
        rec = recall_score(y_test, y_pred)  # Recall for churn class (1)
        f1  = f1_score(y_test, y_pred)

        mlflow.log_param('model', name)
        mlflow.log_metric('accuracy', acc)
        mlflow.log_metric('recall', rec)
        mlflow.log_metric('f1', f1)
        mlflow.sklearn.log_model(model, name.replace(' ', '_'))

        results.append({'Model': name, 'Accuracy': acc, 'Recall': rec, 'F1': f1})
        print(f'{name}: Acc={acc:.4f}, Recall={rec:.4f}, F1={f1:.4f}')

results_df = pd.DataFrame(results).sort_values('Recall', ascending=False)
print('\n--- Ranked by Recall ---')
print(results_df)

## 6. DVC Tracking (run in terminal, not here)
```bash
dvc add ../data/churn_dataset.csv
git add ../data/churn_dataset.csv.dvc ../data/.gitignore
git commit -m "Track churn dataset with DVC"
```

In [ ]:
# View MLflow results
print("MLflow UI: run 'mlflow ui' in terminal, then open http://localhost:5000")
print("\nBest model by Recall:")
print(results_df.iloc[0])